# Qwen Detections

In [8]:
import os
import ast
import json
import pandas as pd
from PIL import Image
from tqdm import tqdm

CSV_FOLDER = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct"
IMAGES_DIR = r"C:\Users\dnnxl\Downloads\Pineapple Video.v2i.coco\train"
OUTPUT_FOLDER = os.path.join(CSV_FOLDER, "coco_annotations")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 🔒 Fixed category definition
CATEGORIES = [
    {
        "id": 0,
        "name": "objects",
        "supercategory": "none"
    },
    {
        "id": 1,
        "name": "pineapple",
        "supercategory": "Crop"
    }
]

# Map label → category_id
LABEL_TO_CAT_ID = {
    "pineapple": 1
}


def convert_csv_to_coco(csv_path):
    df = pd.read_csv(csv_path)

    images = []
    annotations = []

    annotation_id = 1
    image_id = 1

    for _, row in tqdm(df.iterrows(), total=len(df), desc=os.path.basename(csv_path)):
        filename = row["filename"]
        detections = ast.literal_eval(row["raw_detections"])

        image_path = os.path.join(IMAGES_DIR, filename)
        if not os.path.exists(image_path):
            print(f"⚠️ Missing image: {filename}")
            continue

        with Image.open(image_path) as img:
            width, height = img.size

        images.append({
            "id": image_id,
            "file_name": filename,
            "width": width,
            "height": height
        })

        for det in detections:
            x1, y1, x2, y2 = det["bbox_2d"]
            label = det["label"]

            if label not in LABEL_TO_CAT_ID:
                print(f"⚠️ Unknown label '{label}' — skipping")
                continue

            coco_bbox = [x1, y1, x2 - x1, y2 - y1]
            area = (x2 - x1) * (y2 - y1)

            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": LABEL_TO_CAT_ID[label],
                "bbox": coco_bbox,
                "area": area,
                "iscrowd": 0
            })

            annotation_id += 1

        image_id += 1

    coco_output = {
        "images": images,
        "annotations": annotations,
        "categories": CATEGORIES
    }

    out_name = os.path.splitext(os.path.basename(csv_path))[0] + ".json"
    out_path = os.path.join(OUTPUT_FOLDER, out_name)

    with open(out_path, "w") as f:
        json.dump(coco_output, f, indent=2)

    print(f"✅ Saved: {out_path}")


# 🔁 Process all CSV files
for file in os.listdir(CSV_FOLDER):
    if file.endswith(".csv"):
        convert_csv_to_coco(os.path.join(CSV_FOLDER, file))


test_fold_0.csv:   0%|          | 0/308 [00:00<?, ?it/s]

test_fold_0.csv: 100%|██████████| 308/308 [00:00<00:00, 1400.42it/s]


✅ Saved: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_annotations\test_fold_0.json


test_fold_1.csv: 100%|██████████| 308/308 [00:00<00:00, 2223.87it/s]

✅ Saved: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_annotations\test_fold_1.json


...

✅ Saved: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_annotations\test_fold_2.json


test_fold_3.csv: 100%|██████████| 307/307 [00:00<00:00, 2191.16it/s]


✅ Saved: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_annotations\test_fold_3.json


test_fold_4.csv: 100%|██████████| 307/307 [00:00<00:00, 1721.67it/s]

✅ Saved: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_annotations\test_fold_4.json


# Qwen COCO Results Format

https://arena.jamk.fi/en/arena-pro/qwen-vl-vlms-for-zero-and-few-shot-object-detection/

In [13]:
import os
import ast
import json
import pandas as pd
from tqdm import tqdm

CSV_FOLDER = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct"
GT_FOLDER = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds"
OUTPUT_FOLDER = os.path.join(CSV_FOLDER, "coco_predictions")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

LABEL_TO_CAT_ID = {
    "pineapple": 1
}

# 🔧 Scaling factors (from 1000x1000 → 512x512)
SCALE_X = 512 / 1000
SCALE_Y = 512 / 1000
TARGET_SIZE = 512


def load_gt_mapping(gt_json_path):
    with open(gt_json_path, "r") as f:
        gt_data = json.load(f)

    return {
        img["file_name"]: img["id"]
        for img in gt_data["images"]
    }


def convert_csv_to_coco_predictions(csv_path):
    fold_name = os.path.splitext(os.path.basename(csv_path))[0]
    gt_json_path = os.path.join(GT_FOLDER, fold_name + ".json")

    if not os.path.exists(gt_json_path):
        print(f"❌ GT file not found for {fold_name}, expected: {gt_json_path}")
        return

    print(f"\n🔄 Processing {fold_name}")
    print(f"   Using GT: {gt_json_path}")

    filename_to_gt_id = load_gt_mapping(gt_json_path)
    df = pd.read_csv(csv_path)
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=fold_name):
        filename = row["filename"]
        detections = ast.literal_eval(row["raw_detections"])

        if filename not in filename_to_gt_id:
            print(f"⚠️ Image not in GT, skipping: {filename}")
            continue

        image_id = filename_to_gt_id[filename]

        for det in detections:
            x1, y1, x2, y2 = det["bbox_2d"]
            label = det["label"]
            score = float(det.get("score", 1.0))

            if label not in LABEL_TO_CAT_ID:
                print(f"⚠️ Unknown label '{label}' — skipping")
                continue

            # 🔄 Scale coordinates
            x1 *= SCALE_X
            x2 *= SCALE_X
            y1 *= SCALE_Y
            y2 *= SCALE_Y

            # 📏 Clamp to 512×512 bounds
            x1 = max(0, min(TARGET_SIZE, x1))
            y1 = max(0, min(TARGET_SIZE, y1))
            x2 = max(0, min(TARGET_SIZE, x2))
            y2 = max(0, min(TARGET_SIZE, y2))

            # 🧱 Convert to COCO format [x, y, width, height]
            width = max(0, x2 - x1)
            height = max(0, y2 - y1)

            if width == 0 or height == 0:
                continue  # skip invalid boxes

            coco_bbox = [x1, y1, width, height]

            results.append({
                "image_id": image_id,
                "category_id": LABEL_TO_CAT_ID[label],
                "bbox": coco_bbox,
                "score": score
            })

    out_path = os.path.join(OUTPUT_FOLDER, fold_name + "_results.json")

    with open(out_path, "w") as f:
        json.dump(results, f)

    print(f"✅ Saved {len(results)} detections → {out_path}")


# 🔁 Process all CSV folds
for file in os.listdir(CSV_FOLDER):
    if file.endswith(".csv"):
        convert_csv_to_coco_predictions(os.path.join(CSV_FOLDER, file))




🔄 Processing test_fold_0
   Using GT: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_0.json


test_fold_0: 100%|██████████| 308/308 [00:00<00:00, 5578.52it/s]


✅ Saved 1787 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_0_results.json

🔄 Processing test_fold_1
   Using GT: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_1.json


test_fold_1: 100%|██████████| 308/308 [00:00<00:00, 6458.32it/s]


✅ Saved 1722 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_1_results.json

🔄 Processing test_fold_2
   Using GT: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_2.json


...


🔄 Processing test_fold_3
   Using GT: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_3.json


test_fold_3: 100%|██████████| 307/307 [00:00<00:00, 3905.86it/s]


✅ Saved 1686 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_3_results.json

🔄 Processing test_fold_4
   Using GT: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_4.json


test_fold_4: 100%|██████████| 307/307 [00:00<00:00, 5915.48it/s]

✅ Saved 1714 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_4_results.json


# SAM 3 COCO Dataset Format Detections

In [2]:
import os
import ast
import json
import pandas as pd
from tqdm import tqdm

CSV_FOLDER = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct"
GT_FOLDER = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds"
OUTPUT_FOLDER = os.path.join(CSV_FOLDER, "coco_predictions")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

LABEL_TO_CAT_ID = {"pineapple": 1}


def load_gt_info(gt_json_path):
    with open(gt_json_path, "r") as f:
        gt_data = json.load(f)

    filename_to_info = {}
    for img in gt_data["images"]:
        filename_to_info[img["file_name"]] = {
            "id": img["id"],
            "width": img["width"],
            "height": img["height"]
        }
    return filename_to_info


def rescale_bbox_1000_to_image(bbox, img_w, img_h):
    """Convert bbox from 1000x1000 scale to real image size"""
    x1, y1, x2, y2 = bbox
    scale_x = img_w / 1000.0
    scale_y = img_h / 1000.0

    x1 *= scale_x
    x2 *= scale_x
    y1 *= scale_y
    y2 *= scale_y

    return [x1, y1, x2 - x1, y2 - y1]


def convert_csv_to_coco_predictions(csv_path):
    fold_name = os.path.splitext(os.path.basename(csv_path))[0]
    gt_json_path = os.path.join(GT_FOLDER, fold_name + ".json")

    if not os.path.exists(gt_json_path):
        print(f"❌ GT file not found for {fold_name}")
        return

    print(f"\n🔄 Processing {fold_name}")
    filename_to_info = load_gt_info(gt_json_path)

    df = pd.read_csv(csv_path)
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=fold_name):
        filename = row["filename"]
        detections = ast.literal_eval(row["raw_detections"])

        if filename not in filename_to_info:
            print(f"⚠️ Image not in GT, skipping: {filename}")
            continue

        img_info = filename_to_info[filename]
        image_id = img_info["id"]
        img_w = img_info["width"]
        img_h = img_info["height"]

        for det in detections:
            label = det["label"]
            score = float(det.get("score", 1.0))

            if label not in LABEL_TO_CAT_ID:
                continue

            # 🔄 RESCALE HERE
            coco_bbox = rescale_bbox_1000_to_image(det["bbox_2d"], img_w, img_h)

            results.append({
                "image_id": image_id,
                "category_id": LABEL_TO_CAT_ID[label],
                "bbox": coco_bbox,
                "score": score
            })

    out_path = os.path.join(OUTPUT_FOLDER, fold_name + "_predictions.json")

    with open(out_path, "w") as f:
        json.dump(results, f)

    print(f"✅ Saved {len(results)} detections → {out_path}")


# 🔁 Run for all folds
for file in os.listdir(CSV_FOLDER):
    if file.endswith(".csv"):
        convert_csv_to_coco_predictions(os.path.join(CSV_FOLDER, file))




🔄 Processing test_fold_0


test_fold_0: 100%|██████████| 308/308 [00:00<00:00, 5639.52it/s]


✅ Saved 1787 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_0_predictions.json

🔄 Processing test_fold_1


test_fold_1: 100%|██████████| 308/308 [00:00<00:00, 4861.00it/s]

✅ Saved 1722 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_1_predictions.json


...

✅ Saved 1764 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_2_predictions.json

🔄 Processing test_fold_3


test_fold_3: 100%|██████████| 307/307 [00:00<00:00, 2368.12it/s]


✅ Saved 1686 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_3_predictions.json

🔄 Processing test_fold_4


test_fold_4: 100%|██████████| 307/307 [00:00<00:00, 5031.56it/s]

✅ Saved 1714 detections → C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_4_predictions.json


# SAM 3 COCO Results format

In [7]:
import pandas as pd
import json
import ast
import os
from pathlib import Path

CSV_FOLDER = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\SAM3"
GT_BASE = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds"
OUTPUT_DIR = os.path.join(CSV_FOLDER, "coco_results")

CATEGORY_ID = 1  # pineapple

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

for fold in range(5):  # 0–4
    print(f"\n===== Processing FOLD {fold} =====")

    gt_json_path = os.path.join(GT_BASE, f"test_fold_{fold}.json")

    with open(gt_json_path, "r") as f:
        gt_data = json.load(f)

    filename_to_gt_id = {
        img["file_name"]: img["id"]
        for img in gt_data["images"]
    }

    # Only CSVs for this fold
    csv_files = sorted(Path(CSV_FOLDER).glob(f"*fold_{fold}*.csv"))

    if not csv_files:
        print(f"⚠ No CSV files found for fold {fold}")
        continue

    for csv_path in csv_files:
        print(f"Processing {csv_path.name}")

        df = pd.read_csv(csv_path)
        results = []

        for _, row in df.iterrows():
            filename = row["filename"]
            detections = ast.literal_eval(row["detections"])
            scores = ast.literal_eval(row["scores"])

            if filename not in filename_to_gt_id:
                continue

            image_id = filename_to_gt_id[filename]

            for bbox, score in zip(detections, scores):
                x, y, w, h = bbox

                results.append({
                    "image_id": image_id,
                    "category_id": CATEGORY_ID,
                    "bbox": [float(x), float(y), float(w), float(h)],
                    "score": float(score)
                })

        output_path = Path(OUTPUT_DIR) / f"{csv_path.stem}_results.json"
        with open(output_path, "w") as f:
            json.dump(results, f)

        print(f"   → Saved {output_path.name} | Detections: {len(results)}")

print("\n✅ Done converting all folds to COCO results format.")



===== Processing FOLD 0 =====
Processing test_fold_0.csv
   → Saved test_fold_0_results.json | Detections: 1730

===== Processing FOLD 1 =====
Processing test_fold_1.csv
   → Saved test_fold_1_results.json | Detections: 1828

===== Processing FOLD 2 =====
Processing test_fold_2.csv
   → Saved test_fold_2_results.json | Detections: 1778

===== Processing FOLD 3 =====
Processing test_fold_3.csv
   → Saved test_fold_3_results.json | Detections: 1739

===== Processing FOLD 4 =====
Processing test_fold_4.csv
   → Saved test_fold_4_results.json | Detections: 1686

✅ Done converting all folds to COCO results format.


In [12]:
import json

def convert_predictions_to_coco_results(gt_json_path, pred_json_path, output_json_path):
    """
    Convert a COCO-style prediction file into COCO detection results format
    while fixing image_id mismatch using file_name mapping.
    """

    # Load files
    with open(gt_json_path, "r") as f:
        gt_data = json.load(f)

    with open(pred_json_path, "r") as f:
        pred_data = json.load(f)

    # Build filename -> GT image_id map
    filename_to_gt_id = {
        img["file_name"]: img["id"]
        for img in gt_data["images"]
    }

    # Build prediction image_id -> filename map
    pred_id_to_filename = {
        img["id"]: img["file_name"]
        for img in pred_data["images"]
    }

    results = []

    for ann in pred_data["annotations"]:
        pred_image_id = ann["image_id"]
        filename = pred_id_to_filename[pred_image_id]

        if filename not in filename_to_gt_id:
            continue  # skip images not in GT

        result = {
            "image_id": filename_to_gt_id[filename],  # FIXED ID
            "category_id": ann["category_id"],
            "bbox": ann["bbox"],
            "score": ann.get("score", 1.0)
        }

        # Include segmentation if present (for mask evaluation)
        if "segmentation" in ann:
            result["segmentation"] = ann["segmentation"]

        results.append(result)

    # Save results
    with open(output_json_path, "w") as f:
        json.dump(results, f)

    print(f"Saved {len(results)} detections to {output_json_path}")


gt_path = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_0.json"
pred_input = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\SAM3\coco_annotations\test_fold_0.json"
pred_output = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\SAM3\coco_annotations\test_fold_0_results.json"

convert_predictions_to_coco_results(gt_path, pred_input, pred_output)


Saved 1730 detections to C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\SAM3\coco_annotations\test_fold_0_results.json
